# Orbitoby v0.1.1 — Science Day on Google Colab

이 notebook은 **repository clone 없이** PyPI에서 Orbitoby를 설치해
v0.1.0에서 검증한 Science Day workflow를 실행합니다.

필수 Colab Secrets:
- `SPACETRACK_USERNAME`
- `SPACETRACK_PASSWORD`

연구 로직은 `examples/science_day.py`와 동일하게 유지합니다.

In [ ]:
import json
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

CORE = ("numpy", "duckdb", "pandas", "pyarrow", "python-dotenv", "pytz", "requests")
baseline = {}

for package in CORE:
    try:
        baseline[package] = version(package)
    except PackageNotFoundError:
        baseline[package] = None

Path("/content/orbitoby-colab-baseline.json").write_text(
    json.dumps(baseline, indent=2) + "\n",
    encoding="utf-8",
)

print("Before Orbitoby:")
for package, value in baseline.items():
    print(f"{package:14} {value}")

In [ ]:
%pip install "orbitoby[models]==0.1.1"

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path

import orbitoby

baseline = json.loads(
    Path("/content/orbitoby-colab-baseline.json").read_text(encoding="utf-8")
)

print("After Orbitoby:")
for package, before in baseline.items():
    after = version(package)
    print(f"{package:14} before={before!s:10} after={after}")
    if before is not None:
        assert after == before, f"{package} changed: {before} -> {after}"

assert orbitoby.__version__ == "0.1.1"
print("COLAB INSTALL CONTRACT: PASS")

In [ ]:
import os

from google.colab import userdata

for key in ("SPACETRACK_USERNAME", "SPACETRACK_PASSWORD"):
    value = userdata.get(key)
    if not value:
        raise RuntimeError(f"Missing Colab Secret: {key}")
    os.environ[key] = value

print("Space-Track secrets loaded without printing values.")

In [ ]:
import json
from pathlib import Path

import pandas as pd

from orbitoby import Archive, ProductParent
from orbitoby.auth import CredentialManager

START, END = "2024-05-10", "2024-05-11"
METRIC = "thermosphere_neutral_mass_density"
OUTPUT = Path("/content/orbitoby-science-day")
OUTPUT.mkdir(parents=True, exist_ok=True)


def population(archive):
    archive.fetch(source="celestrak", dataset="satcat", all=True)
    rows = archive.objects(
        start=START, end=END, existence="throughout", object_type="payload"
    )
    if rows.empty:
        raise ValueError("empty period population")
    return rows


def historical(archive, candidates):
    selected = candidates[candidates["norad_id"] == 39452]
    if selected.empty:
        raise ValueError("Swarm A absent from period population")

    rows = archive.orbit(norad_id=39452, start=START, end=END)

    if rows.empty or rows["artifact_id"].isna().any():
        raise ValueError("missing historical orbit/provenance")

    summary = archive.orbit_summary(
        selected,
        start=START,
        end=END,
        periapsis_km=(200, 2000),
        orbit_match="all",
        sync=False,
    )

    if not summary["matches"].all():
        raise ValueError("historical orbit screening failed")

    return summary


def density_comparison(archive):
    observed = archive.timeseries(
        METRIC,
        source="swarm",
        dataset="density_a_pod",
        start=START,
        end=END,
    )

    if observed.empty or observed["artifact_id"].isna().any():
        raise ValueError("missing observed density/provenance")

    context = pd.DataFrame(observed["context"].tolist())

    trajectory = pd.DataFrame(
        {
            "timestamp": observed["timestamp"],
            "longitude_deg": context["Longitude_GD"],
            "latitude_deg": context["Latitude_GD"],
            "altitude_km": context["Height_GD"] / 1000.0,
        }
    )

    parents = [
        ProductParent("raw_artifact", str(value), "trajectory_input")
        for value in observed["artifact_id"].unique()
    ]

    inputs, lineage = archive.msis_inputs(
        trajectory,
        driver_source="gfz",
        trajectory_parents=parents,
    )

    modeled = archive.msis_density(inputs, parents=lineage)

    if set(modeled["artifact_kind"]) != {"model_output"}:
        raise ValueError("model output separation lost")

    comparison = observed.merge(
        modeled,
        on="timestamp",
        how="outer",
        suffixes=("_observed", "_model"),
        validate="one_to_one",
        indicator=True,
    )

    if not comparison["_merge"].eq("both").all():
        raise ValueError("unmatched exact timestamps")

    for product_id in modeled["product_id"].unique():
        if not archive.product_store.lineage(product_id):
            raise ValueError("missing model lineage")

    print(
        f"  exact_matches={len(comparison)} "
        f"observed_missing={observed['is_missing'].sum()} "
        f"model_missing={modeled['is_missing'].sum()}"
    )

    return comparison

In [ ]:
report = {}
manager = CredentialManager()

with Archive(credentials=manager) as archive:

    def stage(name, call):
        try:
            frame = call()

            if frame.empty:
                raise ValueError("empty result")

            frame.to_json(
                OUTPUT / f"{name}.jsonl",
                orient="records",
                lines=True,
                date_format="iso",
                double_precision=15,
            )

            report[name] = {
                "status": "PASS",
                "rows": len(frame),
            }

            print(f"PASS {name}: {len(frame)} rows", flush=True)
            return frame

        except Exception as exc:  # noqa: BLE001
            report[name] = {
                "status": "FAIL",
                "error_type": type(exc).__name__,
            }

            print(
                f"FAIL {name}: {type(exc).__name__}",
                flush=True,
            )

            return None

    candidates = stage(
        "population",
        lambda: population(archive),
    )

    if candidates is not None:
        stage(
            "historical_orbit",
            lambda: historical(archive, candidates),
        )
    else:
        report["historical_orbit"] = {
            "status": "SKIP",
            "reason": "population prerequisite failed",
        }

    for metric, source, dataset in (
        ("f107_observed", "gfz", "f107_observed"),
        ("kp", "gfz", "kp"),
        ("ap", "gfz", "ap"),
        ("ap_daily", "gfz", "ap_daily"),
        ("dst", "wdc_kyoto", "dst_hourly"),
        ("solar_wind_speed", "cdaweb", "omni_hourly"),
    ):
        stage(
            metric,
            lambda m=metric, s=source, d=dataset: archive.timeseries(
                m,
                source=s,
                dataset=d,
                start=START,
                end=END,
            ),
        )

    stage(
        "density_comparison",
        lambda: density_comparison(archive),
    )

(OUTPUT / "report.json").write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8",
)

failed = {name: row for name, row in report.items() if row["status"] != "PASS"}

if failed:
    raise RuntimeError(f"Incomplete Science Day workflow: {failed}")

print()
print("SCIENCE DAY COLAB WORKFLOW: PASS")
print("Output:", OUTPUT)

## 이 PASS가 의미하는 것

이 notebook은 데이터 획득·provenance·MSIS 계산·exact-time comparison 경로를 검증합니다.

최종 Science Day 연구에서는 연구자가 연구기간, 변수, orbital envelope,
위성 cohort, mission/mass/shape/inclination/manoeuvre, QC,
orbital-decay 분석 방법을 별도로 결정해야 합니다.